In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

: 

In [ ]:
data  = load_breast_cancer()
X,y = data.data, data.target
print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
print(f"Features names: {data.feature_names}")
print(f"Class names: {data.target_names}")
print(f"Class distribution: {pd.Series(y).value_counts().to_dict()}")

In [ ]:
colors = ['red' if i == 0 else 'blue' for i in y]
plt.title(f"Data visualization of {data.feature_names[0]}")
plt.scatter(X[:, 0], y,color=colors,alpha=0.5,edgecolor='black')

plt.scatter([], [], c='red', edgecolor='k', label='Malignant(0)')
plt.scatter([], [], c='blue', edgecolor='k',label='Benign(1)')
plt.legend(loc='center right')

plt.xlabel(data.feature_names[0])
plt.ylabel('Malignant or Benign')
plt.show()

Training and Testset Splitting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train Linear SVM


In [ ]:
param_grid_linear = { "C":[0.01,0.1,1,10,100]}
linear_svm = LinearSVC(max_iter=10000,random_state=42)
linear_search = GridSearchCV(linear_svm,param_grid=param_grid_linear, cv=5, scoring='f1')
linear_search.fit(X_train_scaled,y_train)

print("Best C for Linear SVM:", linear_search.best_params_)
print("Best CV F1 score:", linear_search.best_score_)

best_linear_svm = linear_search.best_estimator_

Train RBF Kernel SVM


In [ ]:
param_grid_rbf = {
    "C": [0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, 0.1, 1]
}

rbf_svm = SVC(kernel="rbf", random_state=42)
rbf_search = GridSearchCV(rbf_svm, param_grid_rbf, cv=5, scoring="f1")
rbf_search.fit(X_train_scaled, y_train)

print("Best params for RBF SVM:", rbf_search.best_params_)
print("Best CV F1 score:", rbf_search.best_score_)

best_rbf_svm = rbf_search.best_estimator_

Train Logistic Regression Baseline


In [ ]:
logreg = LogisticRegression(max_iter=10000, random_state=42)
logreg.fit(X_train_scaled, y_train)

print("Logistic Regression trained.")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

models = {
    "Linear SVM": best_linear_svm,
    "RBF SVM": best_rbf_svm,
    "Logistic Regression": logreg
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1})

    print("----------------------------------")
    print(f"Performance Of {name} ")
    print("----------------------------------")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

print(" FINAL COMPARISON TABLE")
metrics_df = pd.DataFrame(results)
print(metrics_df.to_string(index=False))

Plot the Confusion matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_model_name = metrics_df.sort_values("F1", ascending=False).iloc[0]["Model"]
best_model = models[best_model_name]

y_pred_best = best_model.predict(X_test_scaled)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best, display_labels=data.target_names, ax=ax, cmap="Grays"
)
ax.set_title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig("../outputs/confusion_matrix.png", dpi=150)
plt.show()

print("Best model based on F1:", best_model_name)

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Reduce to 2 dimensions for visualization only
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Train a fresh RBF SVM on the 2D data (best params from earlier, or defaults)
vis_svm = SVC(kernel="rbf", C=rbf_search.best_params_["C"], gamma=rbf_search.best_params_["gamma"])
vis_svm.fit(X_train_pca, y_train)

# Create a mesh grid over the PCA space
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

Z = vis_svm.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))

# Decision boundary and margins
ax.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=["--", "-", "--"], colors="black")
ax.contourf(xx, yy, Z, levels=[Z.min(), 0, Z.max()], alpha=0.15, colors=["orange", "blue"])

# Data points
scatter = ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap="coolwarm", edgecolors="k", s=30)

# Highlight support vectors
ax.scatter(
    vis_svm.support_vectors_[:, 0], vis_svm.support_vectors_[:, 1],
    facecolors="none", edgecolors="black", s=120, linewidths=1.5, label="Support vectors"
)

ax.set_xlabel("PCA Component 1")
ax.set_ylabel("PCA Component 2")
ax.set_title("RBF SVM Decision Boundary (PCA-reduced to 2D)")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/decision_boundary.png", dpi=150)
plt.show()

print("Number of support vectors:", len(vis_svm.support_vectors_))